In [0]:
CREATE DATABASE NETFLIX;

USE NETFLIX;

CREATE TABLE CUSTOMERS(
    customer_id INT,
    customer_name STRING,
    age INT,
    gender STRING,
    country STRING,
    signup_date DATE
);

INSERT INTO CUSTOMERS
VALUES 
('1001','John Smith','25','Male','USA','2024-01-15'),
('1002','Sarah Jones','31','Female','UK','2024-02-10'),
('1003','Mike Brown','42','Male','Canada','2024-01-22'),
('1004','Emma Wilson','28','Female','USA','2024-03-01'),
('1005','David Lee','36','Male','Australia','2024-02-15'),
('1006','Lisa Green','22','Female','South Africa','2024-04-01'),
('1007','James White','45','Male','UK','2024-01-05'),
('1008','Anna Taylor','33','Female','Canada','2024-03-18');

SELECT *
FROM CUSTOMERS;

CREATE OR REPLACE TABLE Subscription_Plans(
    plan_id STRING,
    plan_name STRING,
    monthly_price INT
);

INSERT INTO Subscription_Plans
VALUES
('P1','Basic','99'),          
('P2','Standard','149'),          
('P3','Premium','199');   

SELECT *
FROM Subscription_Plans;

CREATE OR REPLACE TABLE Subscriptions(
    subscription_id STRING,
    customer_id INT,
    plan_id STRING,
    start_date DATE
);

INSERT INTO Subscriptions
VALUES
('S001','1001','P1','2024-01-15'),
('S002','1002','P2','2024-02-10'),
('S003','1003','P3','2024-01-22'),
('S004','1004','P2','2024-03-01'),
('S005','1005','P1','2024-02-15'),
('S006','1006','P3','2024-04-01'),
('S007','1007','P2','2024-01-05'),
('S008','1008','P3','2024-03-18');

SELECT *
FROM Subscriptions;

CREATE TABLE Viewing_Activity(
    view_id STRING,
    customer_id INT,
    watch_date DATE,
    hours_watched INT 
);


INSERT INTO Viewing_Activity
VALUES
('V001','1001','2026-06-01','3'),
('V002','1001','2026-06-05','5'),            
('V003','1002','2026-06-02','1'),             
('V004','1003','2026-06-01','6'),             
('V005','1004','2026-06-03','2'),             
('V006','1005','2026-06-04','0'),             
('V007','1006','2026-06-02','8'),             
('V008','1007','2026-06-01','4'),             
('V009','1008','2026-06-05','7'),             
('V010','1002','2026-06-07','2'),             
('V011','1004','2026-06-08','3'),            
('V012','1006','2026-06-09','5');             


SELECT *
FROM workspace.netflix.viewing_activity;

-------------------------
SOLUTIONS
-------------------------
1. Display all customers from South Africa.

SELECT *
FROM workspace.netflix.customers
WHERE COUNTRY IN ('South Africa');

2. Display all Premium subscribers.

SELECT plan_id,
       customer_id 
FROM workspace.netflix.subscriptions
WHERE plan_id = 'P3';


3. Show customer names and countries

SELECT customer_name, country
from workspace.netflix.customers;

4. Calculate the age of the oldest customer.

SELECT MAX(Age) AS oldest_cust
from workspace.netflix.customers;

5. Count the total number of customers.

SELECT COUNT(customer_id) AS total_cust
from  workspace.netflix.customers;

6. Join Customers and Subscriptions.

SELECT A.customer_name,
       A.country,
       B.plan_id,
       B.start_date
FROM workspace.netflix.customers AS A
INNER JOIN
workspace.netflix.subscriptions AS B
ON A.customer_id = B.customer_id;

7. Join all three tables.

SELECT A.customer_name,
       A.country,
       C.plan_name,
       B.customer_id,
       C.monthly_price
FROM workspace.netflix.customers AS A
INNER JOIN 
workspace.netflix.subscriptions AS B
ON A.customer_id= B.customer_id
INNER JOIN
workspace.netflix.subscription_plans AS C
ON C.plan_id= B.plan_id;

8. Find total revenue per subscription plan.

SELECT SUM(monthly_price) AS total_revenue, 
       plan_name
from workspace.netflix.subscription_plans
INNER JOIN 
workspace.netflix.subscriptions
ON workspace.netflix.subscriptions.plan_id= workspace.netflix.subscription_plans.plan_id
GROUP BY plan_name;

9. Find the average customer age by country.

SELECT AVG(Age) as avg_age,
       country
FROM workspace.netflix.customers
GROUP BY country;

10. Find the number of subscribers per country

SELECT COUNT(*) AS total_subs, --- I was getting a lot of errors due to the 'COUNT(customer_id)', it was refusing to run because of ambiguity. Now that I said SELECT COUNT(*) it worked because * counts all the rows, whereas 'COUNT(Customer_id)' counts just the customer id.
       country
FROM workspace.netflix.subscriptions
INNER JOIN
workspace.netflix.customers
ON workspace.netflix.customers.customer_id= workspace.netflix.subscriptions.customer_id
GROUP BY country;

11. Find total viewing hours per customer.

SELECT A.customer_name,
       B.hours_watched
FROM workspace.netflix.viewing_activity AS B
INNER JOIN
workspace.netflix.customers AS A
ON A.customer_id= B.customer_id;

12. Find the top 3 most active viewers.

SELECT A.customer_name,
       B.hours_watched
FROM workspace.netflix.viewing_activity AS B
INNER JOIN
workspace.netflix.customers AS A
ON A.customer_id= B.customer_id
ORDER BY hours_watched DESC
LIMIT 3;

13. Create a customer activity classification

SELECT customer_name,
CASE
    WHEN hours_watched >= 10 THEN 'Highly Active'
    WHEN hours_watched >= 5 THEN 'Moderately Active'
    ELSE 'Low Activity'
END AS viewing_activity
FROM workspace.netflix.viewing_activity
hours_watchedINNER JOIN
workspace.netflix.customers
ON workspace.netflix.customers.customer_id = workspace.netflix.viewing_activity.customer_id;

14. Find customers who have never watched any content.
Hint:
LEFT JOIN
IS NULL

SELECT customer_name,
       view_id,
       watch_date,
       hours_watched       
FROM workspace.netflix.viewing_activity
LEFT JOIN
workspace.netflix.customers
ON workspace.netflix.customers.customer_id = workspace.netflix.viewing_activity.customer_id
WHERE hours_watched = '0';
--- David Lee has zero viewing hours.

15. Calculate customer lifetime value estimate.

SELECT customer_name,
       plan_name,
       (monthly_price*12) AS estimated_clv
FROM workspace.netflix.subscription_plans
INNER JOIN
workspace.netflix.subscriptions
ON workspace.netflix.subscriptions.plan_id = workspace.netflix.subscription_plans.plan_id
INNER JOIN
 workspace.netflix.customers
 on workspace.netflix.customers.customer_id = workspace.netflix.subscriptions.customer_id;

 --- I had to join the subscription_plan to the subcriptions on plan_id first because there is no common column between subscription_plan and customers.
 --- Finally join subscription_plan and customers on customer_id.


 -------------------------------------------------
            DASHBOARD QUERY
 -------------------------------------------------
 --- Bar Chart: Subscribers by Country

SELECT COUNT(*) AS number_of_subs,
        country
from workspace.netflix.customers
group by country;

--- Pie Chart:Subscribers by Plan

SELECT COUNT(customer_id) AS subs,
       plan_name
FROM workspace.netflix.subscriptions

INNER JOIN 
workspace.netflix.subscription_plans
ON workspace.netflix.subscriptions.plan_id=workspace.netflix.subscription_plans.plan_id
GROUP BY plan_name;

-- Bar Chart: Revenue by Plan

SELECT SUM(monthly_price) AS total_revenue,
       plan_name
FROM workspace.netflix.subscription_plans
INNER JOIN 
workspace.netflix.subscriptions
ON workspace.netflix.subscriptions.plan_id = workspace.netflix.subscription_plans.plan_id
GROUP BY plan_name;

-- Horizontal Bar Chart: Top 5 Most Active Viewers
SELECT customer_id,
       SUM(hours_watched) AS total_duration
FROM workspace.netflix.viewing_activity
GROUP BY customer_id 
ORDER BY total_duration DESC
LIMIT 5;

-- Bar chart: Viewing activity segements
SELECT customer_name,
CASE
    WHEN hours_watched >= 10 THEN 'Highly Active'
    WHEN hours_watched >= 5 THEN 'Moderately Active'
    ELSE 'Low Activity'
END AS viewing_activity
FROM workspace.netflix.viewing_activity
hours_watchedINNER JOIN
workspace.netflix.customers
ON workspace.netflix.customers.customer_id = workspace.netflix.viewing_activity.customer_id;



---------------------------------------------------------------------
--FINAL EXECUTION SUMMARY--
---------------------------------------------------------------------

WITH CTE1 AS
(
SELECT DISTINCT                ---A=Viewing acticity B=viewing_act C=sub plans D=subscriptions
       A.customer_name, 
       A.country,
      
       B.customer_id,
       D.plan_name,
       D.monthly_price,
       C.start_date,

DAY(start_date) AS day_of_sub,
DAYNAME(start_date) AS dayname_of_sub,

CASE
    WHEN DAYNAME(start_date) in ('Sat','Sun') THEN 'weekend'
    ELSE 'weekday'
    End AS day_class,

MONTHNAME(start_date) AS month_of_sub,
YEAR(start_date) AS year_of_sub,



SUM(B.hours_watched) AS total_screen_time, 
CASE 
    WHEN hours_watched BETWEEN 0 AND 1 THEN 'low_usage'
    WHEN hours_watched BETWEEN 2 AND 8 THEN 'avg_usage'
    WHEN hours_watched >=9 THEN 'high_usage'
END AS screen_buckets,

A.age,
CASE
    WHEN age BETWEEN 20 AND 25 THEN 'general'
    WHEN age BETWEEN 26 AND 30 THEN 'manager'
    WHEN age >=31 THEN 'exec'
END AS age_group
    
    

FROM workspace.netflix.customers AS A
INNER JOIN workspace.netflix.viewing_activity AS B
ON A.customer_id= B.customer_id
INNER JOIN workspace.netflix.subscriptions AS C
ON C.customer_id = B.customer_id
INNER JOIN workspace.netflix.subscription_plans AS D
ON C.plan_id= D.plan_id
GROUP BY ALL
)

-------------------------------------------------
--
----------------
END OF ANALYSIS
----------------
       